In [1]:
import os
from dotenv import load_dotenv

In [2]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print('No API key was found!')
elif not api_key.startswith('sk-proj-'):
    print('An API key was found, but it doesnt start with "sk-proj-", i.e, it is not an OpenAI API key')
elif api_key.strip() != api_key:
    print('An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them.')
else:
    print('The API key found and looks good so far!')


The API key found and looks good so far!


# CrewAI

In [3]:
import os
from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM

# 1. Explicitly define the language model for clarity.
# CrewAI takes its own LLM object (or a model-name string), not a LangChain chat model.
llm = LLM(model="gpt-4-turbo")

# 2. Define a clear and focused agent
planner_writer_agent = Agent(
    role='Article Planner and Writer',
    goal='Plan and then write a concise, engaging summary on a specified topic.',
    backstory=(
        'You are an expert technical writer and content strategist. '
        'Your strength lies in creating a clear, actionable plan before writing, '
        'ensuring the final summary is both informative and easy to digest.'
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm # Assign the specific LLM to the agent
)

# 3. Define a task with a more structured and specific expected output
topic = "The importance of Reinforcement Learning in AI"
high_level_task = Task(
    description=(
        f"1. Create a bullet-point plan for a summary on the topic: '{topic}'.\n"
        f"2. Write the summary based on your plan, keeping it around 200 words."
    ),
    expected_output=(
        "A final report containing two distinct sections:\n\n"
        "### Plan\n"
        "- A bulleted list outlining the main points of the summary.\n\n"
        "### Summary\n"
        "- A concise and well-structured summary of the topic."
    ),
    agent=planner_writer_agent,
)

# Create the crew with a clear process
crew = Crew(
    agents=[planner_writer_agent],
    tasks=[high_level_task],
    process=Process.sequential,
)

# Execute the task.
# The notebook already runs an event loop, so use the async kickoff.
print("## Running the planning and writing task ##")
result = await crew.kickoff_async()

print("\n\n---\n## Task Result ##\n---")
print(result)

## Running the planning and writing task ##


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Article Planner and Writer                                                                              │
│                                                                                                                 │
│  Task: 1. Create a bullet-point plan for a summary on the topic: 'The importance of Reinforcement Learning in   │
│  AI'.                                                                                                           │
│  2. Write the summary based on your plan, keeping it around 200 words.                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Article Planner and Writer                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Plan                                                                                                       │
│  - Define Reinforcement Learning (RL) and how it fits in the broader field of Artificial Intelligence (AI).     │
│  - Discuss key characteristics of RL - learning through interaction, reward systems.                            │
│  - Highlight its applications across various industries - gaming, autonomous vehicles, robotics, etc.           │
│  - Explain the benefits of using RL - adaptability, efficiency, continuous improvement.                         │
│  - Point out challenges and considerations - dependency on quality data, high computational cost.               │
│  - Mention potential future developments in RL.                                                                 │
│                                                                                                                 │
│  ### Summary                                                                                                    │
│  Reinforcement Learning (RL) is a pivotal technique in Artificial Intelligence (AI) where machines learn by     │
│  interacting with their environment. Distinct from other AI methods, RL is based on reward systems where        │
│  actions leading to favorable outcomes are reinforced, mimicking a trial-and-error learning process.            │
│                                                                                                                 │
│  RL has a comprehensive scope of applications including strategic game systems, where it has outperformed       │
│  human experts in complex games like Go and Chess, autonomous driving, where it contributes to safer            │
│  navigation decisions, and advanced robotics, enhancing robots' ability to interact with unstructured           │
│  environments autonomously.                                                                                     │
│                                                                                                                 │
│  The appeal of RL lies in its adaptability and the capability for systems to optimize their performance         │
│  autonomously through continuous learning. This adaptability translates into higher efficiency and              │
│  effectiveness across various tasks and industries.                                                             │
│                                                                                                                 │
│  However, implementing RL is not devoid of challenges. It heavily depends on the availability of quality data,  │
│  and there is a significant computational demand to simulate environments and perform learning iterations.      │
│                                                                                                                 │
│  Despite these challenges, the future of RL is promising with ongoing research aiming to make RL algorithms     │
│  more efficient and less resource-intensive. Innovations are expected to expand its applicability even          │
│  further, potentially automating more complex decision-making processes and enhancing machine autonomy in       │
│  numerous sectors.                                                                                              │
│                                                        



---
## Task Result ##
---
### Plan
- Define Reinforcement Learning (RL) and how it fits in the broader field of Artificial Intelligence (AI).
- Discuss key characteristics of RL - learning through interaction, reward systems.
- Highlight its applications across various industries - gaming, autonomous vehicles, robotics, etc.
- Explain the benefits of using RL - adaptability, efficiency, continuous improvement.
- Point out challenges and considerations - dependency on quality data, high computational cost.
- Mention potential future developments in RL.

### Summary
Reinforcement Learning (RL) is a pivotal technique in Artificial Intelligence (AI) where machines learn by interacting with their environment. Distinct from other AI methods, RL is based on reward systems where actions leading to favorable outcomes are reinforced, mimicking a trial-and-error learning process.

RL has a comprehensive scope of applications including strategic game systems, where it has outperformed human expe

## Crew AI + Ollama

In [4]:
# Same planning example, running on a local model served by Ollama.
# Prerequisites: install Ollama from https://ollama.com, then pull the model:
#   ollama pull qwen2.5:7b

import os
from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM

# 1. Point CrewAI at the local Ollama server instead of a hosted API.
# CrewAI routes models through LiteLLM, so the model name carries an 'ollama/' prefix.
# Ollama runs locally, so no API key is needed.
OLLAMA_MODEL = "ollama/qwen2.5:7b"  # any model from `ollama list`, e.g. ollama/llama3.1:8b

llm = LLM(model=OLLAMA_MODEL, base_url="http://localhost:11434")

# 2. Define a clear and focused agent
planner_writer_agent = Agent(
    role='Article Planner and Writer',
    goal='Plan and then write a concise, engaging summary on a specified topic.',
    backstory=(
        'You are an expert technical writer and content strategist. '
        'Your strength lies in creating a clear, actionable plan before writing, '
        'ensuring the final summary is both informative and easy to digest.'
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm # Assign the specific LLM to the agent
)

# 3. Define a task with a more structured and specific expected output
topic = "The importance of Reinforcement Learning in AI"
high_level_task = Task(
    description=(
        f"1. Create a bullet-point plan for a summary on the topic: '{topic}'.\n"
        f"2. Write the summary based on your plan, keeping it around 200 words."
    ),
    expected_output=(
        "A final report containing two distinct sections:\n\n"
        "### Plan\n"
        "- A bulleted list outlining the main points of the summary.\n\n"
        "### Summary\n"
        "- A concise and well-structured summary of the topic."
    ),
    agent=planner_writer_agent,
)

# Create the crew with a clear process
crew = Crew(
    agents=[planner_writer_agent],
    tasks=[high_level_task],
    process=Process.sequential,
)

# Execute the task.
# The notebook already runs an event loop, so use the async kickoff.
print("## Running the planning and writing task ##")
result = await crew.kickoff_async()

print("\n\n---\n## Task Result ##\n---")
print(result)

## Running the planning and writing task ##


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Article Planner and Writer                                                                              │
│                                                                                                                 │
│  Task: 1. Create a bullet-point plan for a summary on the topic: 'The importance of Reinforcement Learning in   │
│  AI'.                                                                                                           │
│  2. Write the summary based on your plan, keeping it around 200 words.                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Article Planner and Writer                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Plan                                                                                                       │
│  - Introduction to Reinforcement Learning (RL) and its core concept.                                            │
│  - Applications of RL in various fields of AI.                                                                  │
│  - Advantages and benefits of using RL in AI.                                                                   │
│  - Challenges and limitations faced in implementing RL.                                                         │
│  - Future outlook and potential of RL in AI development.                                                        │
│                                                                                                                 │
│  ### Summary                                                                                                    │
│  Reinforcement Learning (RL) is a fundamental approach in artificial intelligence that empowers machines to     │
│  make decisions through trial and error. In this summary, we explore the importance of reinforcement learning   │
│  in AI by highlighting its applications, advantages, challenges, and future prospects.                          │
│                                                                                                                 │
│  - **Applications of RL**: RL is widely applied in diverse areas such as robotics, gaming, finance, and         │
│  healthcare. For instance, smart robots learn to navigate and perform tasks efficiently, self-learning agents   │
│  play complex games like chess and Go, and financial models predict market trends. In healthcare, RL can help   │
│  optimize treatment plans and improve patient outcomes.                                                         │
│                                                                                                                 │
│  - **Advantages of RL**: The key benefits include its ability to learn autonomously without explicit            │
│  programming or large datasets, its effectiveness in environments with uncertainty, and its potential for       │
│  continuous improvement through interaction.                                                                    │
│                                                                                                                 │
│  - **Challenges and Limitations**: Despite its advantages, RL faces significant hurdles like the need for       │
│  extensive training, the computational complexity, and the difficulty in solving problems with multiple         │
│  interacting agents. However, these challenges are being addressed through advancements in algorithms and       │
│  computational power.                                                                                           │
│                                                                                                                 │
│  - **Future Outlook**: The future of RL is promising, with ongoing research focusing on making RL more          │
│  interpretable, scalable, and applicable to real-world problems. As technology advances, we can expect RL to    │
│  become even more integrated into various AI applications, driving greater innovation and progress.             │
│                                                        



---
## Task Result ##
---
### Plan
- Introduction to Reinforcement Learning (RL) and its core concept.
- Applications of RL in various fields of AI.
- Advantages and benefits of using RL in AI.
- Challenges and limitations faced in implementing RL.
- Future outlook and potential of RL in AI development.

### Summary
Reinforcement Learning (RL) is a fundamental approach in artificial intelligence that empowers machines to make decisions through trial and error. In this summary, we explore the importance of reinforcement learning in AI by highlighting its applications, advantages, challenges, and future prospects.

- **Applications of RL**: RL is widely applied in diverse areas such as robotics, gaming, finance, and healthcare. For instance, smart robots learn to navigate and perform tasks efficiently, self-learning agents play complex games like chess and Go, and financial models predict market trends. In healthcare, RL can help optimize treatment plans and improve patient outcomes.



# OpenAI Deep Research API

In [ ]:
# WARNING: this cell makes a real Deep Research call. It runs many web searches
# and can take several minutes and cost dollars, not cents. o4-mini-deep-research
# is the cheaper of the two deep research models; swap in o3-deep-research below
# for the higher-quality (and more expensive) version used in the book.

from openai import OpenAI

# The API key is read from OPENAI_API_KEY, loaded from .env in the first cells.
# Deep research runs long, so raise the client timeout well above the 600s default.
client = OpenAI(timeout=3600)

DEEP_RESEARCH_MODEL = "o4-mini-deep-research"  # or "o3-deep-research"

# Define the agent's role and the user's research question
system_message = """You are a professional researcher preparing a structured, data-driven report.
Focus on data-rich insights, use reliable sources, and include inline citations."""
user_query = "Research the economic impact of semaglutide on global healthcare systems."

# Create the Deep Research API call
response = client.responses.create(
  model=DEEP_RESEARCH_MODEL,
  input=[
    {
      "role": "developer",
      "content": [{"type": "input_text", "text": system_message}]
    },
    {
      "role": "user",
      "content": [{"type": "input_text", "text": user_query}]
    }
  ],
  reasoning={"summary": "auto"},
  tools=[{"type": "web_search_preview"}]
)

# Access and print the final report from the response
final_report = response.output[-1].content[0].text
print(final_report)

# --- ACCESS INLINE CITATIONS AND METADATA ---
print("--- CITATIONS ---")
annotations = response.output[-1].content[0].annotations

if not annotations:
    print("No annotations found in the report.")
else:
    for i, citation in enumerate(annotations):
        # The text span the citation refers to
        cited_text = final_report[citation.start_index:citation.end_index]

        print(f"Citation {i+1}:")
        print(f"  Cited Text: {cited_text}")
        print(f"  Title: {citation.title}")
        print(f"  URL: {citation.url}")
        print(f"  Location: chars {citation.start_index}–{citation.end_index}")
print("\n" + "="*50 + "\n")


# --- INSPECT INTERMEDIATE STEPS ---
print("--- INTERMEDIATE STEPS ---")

# 1. Reasoning Steps: Internal plans and summaries generated by the model.
try:
    reasoning_step = next(item for item in response.output if item.type == "reasoning")
    print("\n[Found a Reasoning Step]")
    for summary_part in reasoning_step.summary:
        print(f"  - {summary_part.text}")
except StopIteration:
    print("\nNo reasoning steps found.")

# 2. Web Search Calls: The exact search queries the agent executed.
try:
    search_step = next(item for item in response.output if item.type == "web_search_call")
    print("\n[Found a Web Search Call]")
    # action is a model object (ActionSearch), not a dict.
    print(f"  Query Executed: '{search_step.action.query}'")
    print(f"  Status: {search_step.status}")
except StopIteration:
    print("\nNo web search steps found.")

# 3. Code Execution: Any code run by the agent using the code interpreter.
try:
    code_step = next(item for item in response.output if item.type == "code_interpreter_call")
    print("\n[Found a Code Execution Step]")
    print("  Code Input:")
    print(f"  ```python\n{code_step.input}\n  ```")
    print("  Code Output:")
    print(f"  {code_step.output}")
except StopIteration:
    print("\nNo code execution steps found.")
